In [ ]:
#Inicializar rejilla 
#Medidas en metros
import numpy as np
ancho_recinto = 600
alto_recinto = 200

muros = np.zeros((alto_recinto,ancho_recinto),dtype = bool)

def pintar(mascara, x, y, ancho, fondo):
    mascara[y:y + fondo, x:x + ancho] = True



import matplotlib.pyplot as plt

plt.imshow(muros, origin="lower", extent=(0, ancho_recinto, 0, alto_recinto), cmap="Greys")
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.show()


In [ ]:
W, H = 200, 120
muros = np.zeros((H, W), dtype=bool)
def abrir(mascara, x, y, ancho, fondo):
    mascara[y:y + fondo, x:x + ancho] = False

# vestíbulo al sur, separado de las salas por un muro con cuatro puertas de 8 m
pintar(muros, 0, 12, W, 1)
for x in (20, 60, 120, 170):
    abrir(muros, x, 12, 8, 1)

# taquillas que estrechan el vestíbulo
pintar(muros, 80, 4, 30, 2)

# muro que divide la sala A (oeste) de la sala B (este), con dos pasos de 6 m
pintar(muros, 110, 13, 1, H - 13)
abrir(muros, 110, 40, 1, 6)
abrir(muros, 110, 85, 1, 6)

# sala A: columnas de 1 × 1 m cada 12 m
for x in range(12, 110, 12):
    for y in range(25, H, 12):
        pintar(muros, x, y, 1, 1)

# bodega cerrada en la esquina noroeste, con una sola puerta de 2 m
pintar(muros, 0, 95, 25, 1)
pintar(muros, 25, 95, 1, H - 95)
abrir(muros, 10, 95, 2, 1)

# sala B: escenario al fondo y barra pegada al muro este
pintar(muros, 130, 105, 50, 15)
pintar(muros, 194, 40, 6, 20)
salidas = np.zeros((H, W), dtype=bool)
pintar(salidas, 30, 0, 10, 1)        # sur, entrada principal
pintar(salidas, 140, 0, 10, 1)       # sur, segunda entrada
pintar(salidas, 0, 50, 1, 6)         # oeste, sala A
pintar(salidas, 60, H - 1, 6, 1)     # norte, sala A
pintar(salidas, W - 1, 70, 1, 6)     # este, sala B
pintar(salidas, 185, H - 1, 6, 1)    # norte, sala B, junto al escenario
from matplotlib.colors import ListedColormap

plano = np.zeros((H, W), dtype=int)
plano[muros] = 1
plano[salidas] = 2

plt.imshow(plano, origin="lower", extent=(0, W, 0, H), vmin=0, vmax=2,
           cmap=ListedColormap(["white", "black", "limegreen"]))
plt.show()


### Ecuacion Eikonal

$$|\nabla u(x)| = n(x)$$
$n(x)$ es el índice de refracción

$\nabla u$ (el gradiente de $u$) 
|| Norma ( $\nabla u$ devuelve un vector en cada una de las dimensiones, por eso se pone $\nabla u$  y no du  )



### Fast Marching Method Complejidad (MLogM) M = celdas

$$
\begin{aligned}
|\nabla u(x)| &= 1/f(x) \quad \text{for } x \in \Omega \\
u(x) &= 0 \quad \text{for } x \in \partial\Omega
\end{aligned}
$$

u(x) = el valor objetivo = tiempo acumulado, etc
∂Ω distancia acumalada a salidas = cero en las salidas =  frontera de Omega

Ω = terreno

f(x)(equivalente a nx pero no solo para optica) = velocidad en el terreno. Si es constante por ende se obtiene la distancia geodesica de cada punto

### Algoritmo
Se discretiza el dominio 
A cada celda xi se le asigna un valor $$U_i = U(x_i) \approx u(x_i).$$
El algoritmo trabaja igual que Dijkstra pero pesos de los nodos se calculan diferente (esta diferencia es la que hace que este algoritmo soporte diagonales): 
Los nodos no explorados se etiquetan como lejanos y tienen tiempo infinito (esto nos permite saber que areas del recinto son inaccesibles)

Se clasifica como considerado el borde de la "onda de tiempo", son nodos evaluados pero que quizas puedan mejorar
Se clasifican como aceptadas las celdas cuyo tiempo es el minimo por lo que ya no va a cambiar

1. Se inicializan todas las celdas co tiempo infinito y se etiquetan como lejanas
2. Las celdas meta (∂Ω) se les pone tiempo Ui = 0 y se marcan como aceptadas (son el punto de inicio)
3. Estimar vecinos: A las celdas que tocan una meta se les calcula un valor estimado U con la Eikonal. este valor al ser menor a infinito se guarda y se mueve a la lista de considerados
4. Elegir al mejor candidato: Se busca en toda la lista de considerados cual es la celda x que tiene el U mas pequeno de todos, esta celda pasa a ser etiquetada como aceptada y se congela
5. Se propaga: se aplica el mismo criterio con las nuevas celda que tocan una celda meta

Con esto se obtiene un mapa de tiempos de cada celda, si ademas agregamos otra etiqueta como lo puede ser un id de la celda origen que llega primero obtenemos el mapa de voronoi
Para pasar del mapa de tiempos a la mejor ruta basta con ahcer un descenso del gradiente o backtracking (buscar en las celdas vecinas cual tiene el menor tiempo  o sea la direccion opuesta al gradiente $-\nabla u$)


In [ ]:
%pip install plotly scikit-fmm kaleido

In [ ]:
import skfmm

phi = np.ones((H, W))                         # +1 en todo el recinto
phi[salidas] = -1                             # -1 en las salidas: de ahí arranca el frente
phi = np.ma.MaskedArray(phi, mask=muros)      # las celdas de muro no se recorren

v = np.ma.MaskedArray(np.ones((H, W)), mask=muros)   # velocidad 1 m/s en todas partes

D = skfmm.travel_time(phi, v, dx=1)           # dx = tamaño de celda en metros
D = np.ma.filled(D, np.inf)                   # donde había muro queda infinito


In [ ]:
Dm = np.ma.masked_invalid(D)          # oculta los infinitos al dibujar

plt.imshow(Dm, origin="lower", extent=(0, W, 0, H), cmap="viridis")
plt.colorbar(label="distancia a la salida más cercana (m)")
plt.contour(Dm, levels=[40], colors="red", origin="lower", extent=(0, W, 0, H))
plt.show()

print("punto más alejado:", D[np.isfinite(D)].max(), "m")


In [ ]:
Z = D.copy()
techo = D[np.isfinite(D)].max() * 1.15
Z[~np.isfinite(Z)] = techo                    # muros = paredes altas

X, Y = np.meshgrid(np.arange(W), np.arange(H))   # X[y, x] = x,  Y[y, x] = y

fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.plot_surface(X, Y, Z, cmap="viridis", rstride=2, cstride=2, linewidth=0)


In [ ]:
def canica(D, x, y):
    camino = [(x, y)]
    while True:
        mejor = (D[y, x], x, y)
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                vx, vy = x + dx, y + dy
                if 0 <= vx < W and 0 <= vy < H and D[vy, vx] < mejor[0]:
                    mejor = (D[vy, vx], vx, vy)
        if (mejor[1], mejor[2]) == (x, y):    # ninguna vecina más abajo: llegó
            return camino
        x, y = mejor[1], mejor[2]
        camino.append((x, y))


In [ ]:
import plotly.graph_objects as go

dmax = D[np.isfinite(D)].max()
Z = np.where(np.isfinite(D), D, dmax + 10)        # muros e inalcanzable: más altos que todo

fig = go.Figure()
fig.add_surface(x=np.arange(W), y=np.arange(H), z=Z, colorscale="Viridis", cmin=0, cmax=dmax)

ys, xs = np.nonzero(salidas)                      # salidas: el fondo de los hoyos
fig.add_scatter3d(x=xs, y=ys, z=np.zeros(len(xs)), mode="markers",
                  marker=dict(color="lime", size=3), name="salidas")

for x0, y0 in [(20, 110), (100, 100), (190, 90), (60, 60)]:
    c = canica(D, x0, y0)
    cx, cy = zip(*c)
    fig.add_scatter3d(x=cx, y=cy, z=[D[y, x] + 2 for x, y in c],
                      mode="lines+markers", showlegend=False,
                      line=dict(color="red", width=6), marker=dict(color="red", size=3))

fig.update_layout(scene=dict(aspectmode="manual", aspectratio=dict(x=2, y=1.2, z=0.6)),
                  scene_camera=dict(eye=dict(x=0, y=-0.8, z=2.0)))
fig.show()


In [ ]:
from enum import Enum
from dataclasses import dataclass


class Tipo(Enum):
    #                     nombre              ancho fondo  color     unidades  puerta
    SANITARIO         = ("Sanitarios",          8,   3,  "#4C9BE8",  8,      (0, 8))
    COMIDA            = ("Comida",              5,   5,  "#E8A33D",  4,      (1, 4))
    BEBIDAS           = ("Bebidas",             8,   3,  "#7BC96F",  6,      (0, 8))
    PRIMEROS_AUXILIOS = ("Primeros auxilios",   5,   5,  "#A96BE0",  2,      (2, 3))

    def __init__(self, nombre, ancho, fondo, color, unidades, puerta):
        self.nombre = nombre
        self.ancho = ancho
        self.fondo = fondo
        self.color = color
        self.unidades = unidades
        self.puerta = puerta      # (desde, hasta) en metros sobre el frente

@dataclass
class Modulo:
    tipo: Tipo
    x: int              # esquina inferior izquierda de lo que ocupa en el piso
    y: int
    rot: int = 0        # 0, 90, 180 o 270, en sentido antihorario

    def __post_init__(self):
        assert self.rot in (0, 90, 180, 270), "rot debe ser 0, 90, 180 o 270"

    @property
    def ancho(self):    # lo que ocupa en x, ya girado
        return self.tipo.ancho if self.rot in (0, 180) else self.tipo.fondo

    @property
    def fondo(self):    # lo que ocupa en y, ya girado
        return self.tipo.fondo if self.rot in (0, 180) else self.tipo.ancho

    def celdas_puerta(self):
        a, b = self.tipo.puerta
        x, y, w, f = self.x, self.y, self.ancho, self.fondo
        if self.rot == 0:      # frente al sur: la fila de abajo
            return [(x + u, y - 1) for u in range(a, b)]
        if self.rot == 90:     # frente al este: la columna a la derecha
            return [(x + w, y + u) for u in range(a, b)]
        if self.rot == 180:    # frente al norte: la fila de arriba, recorrida al revés
            return [(x + w - 1 - u, y + f) for u in range(a, b)]
        if self.rot == 270:    # frente al oeste: la columna a la izquierda, recorrida al revés
            return [(x - 1, y + f - 1 - u) for u in range(a, b)]




In [ ]:
layout = [
    Modulo(Tipo.SANITARIO, 14, 28, 0),
    Modulo(Tipo.SANITARIO, 62, 28, 180),
    Modulo(Tipo.SANITARIO, 150, 30, 0),
    Modulo(Tipo.COMIDA, 38, 64, 90),
    Modulo(Tipo.COMIDA, 86, 64, 270),
    Modulo(Tipo.COMIDA, 140, 70, 0),
    Modulo(Tipo.BEBIDAS, 50, 100, 0),
    Modulo(Tipo.BEBIDAS, 160, 95, 180),
    Modulo(Tipo.PRIMEROS_AUXILIOS, 74, 40, 90),
]


In [ ]:
def campo(destino, bloqueado):
    phi = np.ones((H, W))
    phi[destino] = -1
    phi = np.ma.MaskedArray(phi, mask=bloqueado)
    v = np.ma.MaskedArray(np.ones((H, W)), mask=bloqueado)
    return np.ma.filled(skfmm.travel_time(phi, v, dx=1), np.inf)

def mascara_puertas(layout, tipo):
    M = np.zeros((H, W), dtype=bool)
    for m in layout:
        if m.tipo is tipo:
            for px, py in m.celdas_puerta():
                M[py, px] = True
    return M

ocupado = muros.copy()
for m in layout:
    pintar(ocupado, m.x, m.y, m.ancho, m.fondo)

D_san = campo(mascara_puertas(layout, Tipo.SANITARIO), ocupado)


In [ ]:
import numpy as np
import plotly.graph_objects as go

D = campo(salidas, ocupado)              # terreno: distancia a la salida, con los módulos estorbando
print(D.dtype, D[np.isfinite(D)].min(), D[np.isfinite(D)].max())   # debe dar float64, menos de 1 y unos 84
dmax = D[np.isfinite(D)].max()
alto = dmax + 10                         # altura de todo lo que no se puede pisar

cuerpo = np.zeros((H, W), dtype=bool)    # solo los módulos, para dibujarlos aparte
for m in layout:
    pintar(cuerpo, m.x, m.y, m.ancho, m.fondo)

Z = np.where(np.isfinite(D), D, alto)    # muros altos
Z[cuerpo] = np.nan                       # hueco donde va cada módulo: ahí entra su bloque de color

def caja(m, alto):
    x0, x1, y0, y1 = m.x, m.x + m.ancho, m.y, m.y + m.fondo
    return go.Mesh3d(x=[x0, x1, x1, x0, x0, x1, x1, x0],
                     y=[y0, y0, y1, y1, y0, y0, y1, y1],
                     z=[0, 0, 0, 0, alto, alto, alto, alto],
                     alphahull=0, color=m.tipo.color, flatshading=True, name=m.tipo.nombre)

fig = go.Figure()
fig.add_surface(x=np.arange(W) + 0.5, y=np.arange(H) + 0.5, z=Z,
                colorscale="Viridis", cmin=0, cmax=dmax, showscale=False)

for m in layout:
    fig.add_trace(caja(m, alto))
    px, py = zip(*m.celdas_puerta())
    fig.add_scatter3d(x=np.array(px) + 0.5, y=np.array(py) + 0.5,
                      z=[D[y, x] + 1 for x, y in m.celdas_puerta()],
                      mode="markers", marker=dict(color="black", size=3), showlegend=False)

ys, xs = np.nonzero(salidas)
fig.add_scatter3d(x=xs + 0.5, y=ys + 0.5, z=np.zeros(len(xs)), mode="markers",
                  marker=dict(color="lime", size=3), name="salidas")



fig.update_layout(scene=dict(aspectmode="manual", aspectratio=dict(x=2, y=1.2, z=0.6)),
                  scene_camera=dict(eye=dict(x=0, y=-0.8, z=2.0)))
fig.show()


Se esta graficando el mapa de tiempos que se obtiene de la eikonal como la altura en el mapa, por eso los muros que tienen un tiempo infinito quedan inalcanzables y las salidas con tiempo 0, de esta forma queda muy claro visualemnte como el descensodel gradiente nos da la direccion a la salida mas cercana (por eso la canica en una superficie suave)

In [ ]:
V0, GAMMA, RHO_MAX = 1.34, 1.913, 5.4    # Weidmann (1993): m/s, p/m², p/m²

def weidmann(rho):
    rho = np.clip(rho, 1e-6, RHO_MAX)    # sin dividir entre cero ni pasarse del atasco
    return V0 * (1 - np.exp(-GAMMA * (1 / rho - 1 / RHO_MAX)))


In [ ]:
rho = np.linspace(0.01, 5.4, 500)
v = weidmann(rho)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(rho, v);       ax[0].set_xlabel("densidad (p/m²)"); ax[0].set_ylabel("velocidad (m/s)")
ax[1].plot(rho, rho * v); ax[1].set_xlabel("densidad (p/m²)"); ax[1].set_ylabel("flujo (p/m/s)")
plt.show()


In [ ]:
def campo(destino, bloqueado, v=None):
    if v is None:
        v = np.ones((H, W))                       # sin velocidad: distancia en metros
    phi = np.ones((H, W))
    phi[destino] = -1
    phi = np.ma.MaskedArray(phi, mask=bloqueado)
    v = np.ma.MaskedArray(np.maximum(v, 1e-3), mask=bloqueado)
    return np.ma.filled(skfmm.travel_time(phi, v, dx=1), np.inf)


Si ignoramos el componente de altura (la distancia -- tiempo a la salida) obtenemos la ruta mas corta a la salida (o sea la direccion en la que fluye cada celda en la simulacion sin emabrgo la velocidad nos la da e ldiagrama fundamental de weidmann)

### D y T: el terreno sin gente y con gente

`campo()` resuelve la misma eikonal con dos velocidades distintas:

$$\|\nabla u(x)\| = n(x) = \frac{1}{v(x)}$$

| | velocidad $v(x)$ | qué da | se usa en |
|---|---|---|---|
| **D** = `campo(salidas, ocupado)` | 1 en todas partes | metros a la salida más cercana | dibujar, canicas |
| **T** = `campo(salidas, ocupado, weidmann(M))` | Weidmann, según la densidad de cada celda | segundos para salir con la gente que hay **ahora** | `paso()`: hacia dónde va cada quien |

En T la densidad entra como $n(x)$: una celda llena es "lenta" y cruzarla cuesta más tiempo. Como T es el camino de **menor tiempo**, si rodear a la multitud cuesta menos que atravesarla, T ya lo toma en cuenta.

**Modelo de Hughes** (Hughes, 2002): la gente busca el camino más **rápido**, no el más corto, y decide con lo que ve en ese instante. A esa forma de decidir se le llama *reactiva* (*reactive dynamic user equilibrium*; Huang et al., 2009; Jiang et al., 2009).

**Matices**
1. **T es una foto.** Supone que la multitud se queda como está; no prevé que el tapón se disuelva. Por eso se recalcula cada `cada = 8` pasos = 8 × 0.3 s = **2.4 s**. Lo ideal sería en cada paso, pero la eikonal es lo más caro de la simulación.
2. **Una celda vacía también cuesta:** $n = 1/1.34$ s/m. Donde no hay nadie, $T \approx D / V_0$. Al inicio (0.27 p/m², v = 1.339 m/s) el punto más lejano tiene D = 84.3 m y T = 63.0 s ≈ 84.3 / 1.34.
3. **El atasco cuesta mucho, pero no es muro:** cerca de 5.4 p/m², $v \to 0$ y $n \to \infty$. `np.maximum(v, 1e-3)` topa el costo en 1000 s/m; sin eso, una salida atascada quedaría "cerrada" en el mapa y nadie más iría hacia ella.

**Ojo: no sumar D + T.** T ya incluye a D; lo que T agrega es el retraso por la multitud:

$$T = \frac{D}{V_0} + \text{retraso} \qquad\Longrightarrow\qquad V_0\,T = D + V_0\cdot\text{retraso}$$

De aquí salen dos gráficas: el mapa de $V_0 T - D$ (cuánto estorba la multitud, después de la simulación) y el GIF del líquido, que usa D de piso y la densidad como altura extra encima: $D + K\rho$.

**Dos formas en que la gente se estorba** (no confundirlas):
- **Al elegir ruta (T):** evita zonas llenas, hasta puede cambiar de salida. Es global.
- **Al moverse (Weidmann + Godunov, en `paso()`):** dentro de una multitud se camina más lento y no se entra a una celda llena. Es local.

*Hughes, R. L. (2002). A continuum theory for the flow of pedestrians. Transportation Research Part B, 36(6), 507–535.*
*Huang, L., Wong, S. C., Zhang, M., Shu, C.-W. y Lam, W. H. K. (2009). Revisiting Hughes' dynamic continuum model for pedestrian flow and the development of an efficient solution algorithm. Transportation Research Part B, 43(1), 127–141.*
*Jiang, Y. et al. (2009). A reactive dynamic continuum user equilibrium model for bi-directional pedestrian flows. Acta Mathematica Scientia, 29(6), 1541–1555.*

In [ ]:
_r = np.linspace(0.01, RHO_MAX, 100000)
Q_MAX = (_r * weidmann(_r)).max()             # el flujo más alto posible: 1.2249 p/(m·s)
RHO_CAP = _r[(_r * weidmann(_r)).argmax()]    # la densidad donde ocurre: 1.751 p/m²


### `paso()`: Godunov (demanda y oferta)

En cada paso, para cada celda:

- **Demanda** (`envia`), lo que quiere salir: $\rho\,v(\rho)$ si $\rho < \rho_{crit}$; si no, la capacidad $Q_{max}$.
  Aquí está el destrabe: una celda llena no pide $\rho\,v(\rho)$, que tiende a 0, sino la capacidad. Si la de adelante tiene espacio, sale.
- **Oferta** (`recibe`), lo que cabe: $Q_{max}$ si $\rho < \rho_{crit}$; si no, $\rho\,v(\rho)$. En muros y módulos, 0.
- **Flujo = min(demanda, oferta)** (`factor`): si a una celda le quieren meter más de lo que cabe, se recorta a todas las que mandan, en la misma proporción.
- **Reparto** (`caidas ** p`): la demanda de cada celda se reparte entre sus vecinas cuesta abajo en T, con peso $(\Delta T)^p$.

**Sustento:** van Wageningen-Kessels, Daamen y Hoogendoorn (2018) extienden Godunov (Lebacque, 1996) a 2D para peatones: demanda, oferta y flujo = mínimo, con los mismos $v_{libre} = 1.34$ y $\rho_{jam} = 5.4$. Garantizan que la densidad se queda entre 0 y $\rho_{jam}$ (Teorema 2) si se cumple la CFL (Teorema 1): $v_{libre}\,\Delta t \le \Delta x\,\Delta y / \sqrt{\Delta x^2 + \Delta y^2}$. Aquí: 1.34 × 0.3 = 0.40 ≤ 0.71 m.

**En qué difiere este `paso()` del paper:**
- **El reparto con p = 3 es nuestro, sin respaldo.** El paper reparte con las componentes de la dirección de marcha: $\max(0, e_x)$ a la cara este, $\max(0, -e_x)$ a la oeste, y lo mismo en y.
- El paper además topa la demanda con lo que hay en la celda (ec. 14) y la oferta con lo que falta para el atasco (ec. 21). Este `paso()` no lo hace, pero con estos parámetros los dos topes se cumplen solos. El kernel del proyecto (`nucleo/kernel.py`) sí los tiene.

*van Wageningen-Kessels, F., Daamen, W. y Hoogendoorn, S. P. (2018). Two-Dimensional Approximate Godunov Scheme and What It Means For Continuum Pedestrian Flow Models. Transportation Science, 52(3), 547–563. doi:10.1287/trsc.2017.0793*

In [ ]:
def mover(A, dy, dx, relleno=0.0):
    """Corre la matriz una celda: lo que estaba en [y, x] queda en [y+dy, x+dx]."""
    B = np.full_like(A, relleno)
    B[max(dy, 0):H + min(dy, 0), max(dx, 0):W + min(dx, 0)] = \
        A[max(-dy, 0):H + min(-dy, 0), max(-dx, 0):W + min(-dx, 0)]
    return B

DIRECCIONES = [(1, 0), (-1, 0), (0, 1), (0, -1)]    # (dy, dx): norte, sur, este, oeste

def paso(M, T, dt=0.3, p=3):
    q = M * weidmann(M)                              # flujo de Weidmann en cada celda
    envia = np.where(M <= RHO_CAP, q, Q_MAX)         # demanda: lo que quiere salir
    recibe = np.where(M <= RHO_CAP, Q_MAX, q)        # oferta: lo que todavía cabe
    recibe[ocupado] = 0                              # a muros y módulos no entra nadie

    caidas = []
    for dy, dx in DIRECCIONES:
        Tv = mover(T, -dy, -dx, relleno=np.inf)      # T de la vecina; fuera del mapa = infinito
        caida = np.zeros_like(T)
        ok = np.isfinite(T) & np.isfinite(Tv)        # solo entre celdas pisables
        caida[ok] = np.maximum(T[ok] - Tv[ok], 0)    # cuánto baja hacia ella; si sube, cero
        caidas.append(caida ** p)
    total = sum(caidas)
    total[total == 0] = np.inf                       # sin bajada: no manda nada

    quiere = [envia * dt * c / total for c in caidas]                  # reparto cuesta abajo
    llega = sum(mover(f, dy, dx) for f, (dy, dx) in zip(quiere, DIRECCIONES))
    factor = np.minimum(1, recibe * dt / np.maximum(llega, 1e-12))    # si no cabe todo, se recorta

    nuevo = M.copy()
    for f, (dy, dx) in zip(quiere, DIRECCIONES):
        f = f * mover(factor, -dy, -dx)              # lo que de verdad pasa
        nuevo -= f                                   # sale de mi celda
        nuevo += mover(f, dy, dx)                    # entra a la vecina

    sale = np.where(salidas, np.minimum(nuevo, Q_MAX * dt), 0)        # cada metro de puerta drena Q_MAX
    return nuevo - sale, sale.sum()


In [ ]:
def paso_solo_weidmann(M, T, dt=0.3, p=3):
    """Igual que paso(), pero sin demanda ni oferta: cada celda manda densidad × velocidad
    cuesta abajo y la vecina lo recibe todo, quepa o no."""
    q = M * weidmann(M)
    caidas = []
    for dy, dx in DIRECCIONES:
        Tv = mover(T, -dy, -dx, relleno=np.inf)
        caida = np.zeros_like(T)
        ok = np.isfinite(T) & np.isfinite(Tv)
        caida[ok] = np.maximum(T[ok] - Tv[ok], 0)
        caidas.append(caida ** p)
    total = sum(caidas)
    total[total == 0] = np.inf
    nuevo = M.copy()
    for c, (dy, dx) in zip(caidas, DIRECCIONES):
        f = q * dt * c / total                       # todo lo que quiere salir, sale...
        nuevo -= f
        nuevo += mover(f, dy, dx)                    # ...y entra aunque la vecina ya esté llena
    sale = np.where(salidas, np.minimum(nuevo, Q_MAX * dt), 0)
    return nuevo - sale, sale.sum()

def simular(fn, aforo, t_max=900, dt=0.3, cada=8):
    """Evacuación con la regla de paso 'fn'. Regresa personas dentro y densidad máxima en cada paso."""
    M = np.where(~ocupado, aforo / (~ocupado).sum(), 0.0)
    dentro, pico = [], []
    for n in range(int(t_max / dt)):
        if n % cada == 0:
            T = campo(salidas, ocupado, weidmann(M))
        M, _ = fn(M, T, dt)
        dentro.append(M.sum())
        pico.append(M.max())
        if M.sum() <= 0.005 * aforo:
            break
    return np.array(dentro), np.array(pico), M

aforo = 6000
dentro_g, pico_g, _ = simular(paso, aforo)
dentro_w, pico_w, M_w = simular(paso_solo_weidmann, aforo)
t_g = np.arange(len(dentro_g)) * 0.3
t_w = np.arange(len(dentro_w)) * 0.3

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
ax[0].plot(t_g, dentro_g, label="Weidmann + Godunov")
ax[0].plot(t_w, dentro_w, label="solo Weidmann")
ax[0].set_xlabel("tiempo (s)"); ax[0].set_ylabel("personas dentro"); ax[0].legend()

ax[1].plot(t_g, pico_g, label="Weidmann + Godunov")
ax[1].plot(t_w, pico_w, label="solo Weidmann")
ax[1].axhline(RHO_MAX, color="k", ls="--", lw=1, label="atasco: 5.4 p/m²")
ax[1].set_yscale("log")
ax[1].set_xlabel("tiempo (s)"); ax[1].set_ylabel("densidad máxima (p/m²)"); ax[1].legend()

ax[2].imshow(np.where(ocupado, 1.0, np.nan), origin="lower", extent=(0, W, 0, H), cmap="Greys", vmin=0, vmax=1.5)
im = ax[2].imshow(np.where(M_w > 0.01, M_w, np.nan), origin="lower", extent=(0, W, 0, H),
                  cmap="Reds", vmin=0, vmax=RHO_MAX)
ax[2].set_title("solo Weidmann a los %.0f s: dónde se queda la gente" % t_w[-1])
plt.colorbar(im, ax=ax[2], label="p/m² (rojo oscuro: 5.4 o más)")
plt.tight_layout()
plt.show()

print("Weidmann + Godunov: evacúa en %.0f s y la densidad nunca pasa de %.2f p/m²" % (t_g[-1], pico_g.max()))
print("Solo Weidmann: a los %.0f s siguen dentro %.0f personas, y hay celdas con %.0f p/m² (el atasco es 5.4)"
      % (t_w[-1], dentro_w[-1], pico_w.max()))

### Parámetros

| Parámetro | Valor | Qué es | De dónde sale |
|---|---|---|---|
| `V0` ($v_0$) | 1.34 m/s | velocidad libre, sin nadie alrededor | Weidmann (1993) |
| `GAMMA` ($\gamma$) | 1.913 p/m² | qué tan rápido baja la velocidad al llenarse | Weidmann (1993) |
| `RHO_MAX` ($\rho_{max}$) | 5.4 p/m² | densidad de atasco: v = 0 | Weidmann (1993) |
| `RHO_CAP` ($\rho_{crit}$) | 1.751 p/m² | densidad donde el flujo es máximo | se calcula de Weidmann |
| `Q_MAX` | 1.2249 p/(m·s) | capacidad: lo más que pasa por 1 m de ancho en 1 s | se calcula: $\rho_{crit}\,v(\rho_{crit})$ |
| `dt` ($\Delta t$) | 0.3 s | paso de tiempo | CFL 2D: 0.40 ≤ 0.71 m; el máximo sería 0.53 s |
| `cada` | 8 pasos = 2.4 s | cada cuánto se recalcula T | nuestro; medido en el proyecto (`REFRESCO_CAMPO`) |
| `p` | 3 | reparto entre vecinas cuesta abajo | nuestro, **sin respaldo** (ver nota de `paso()`) |
| `1e-3` en `campo()` | 0.001 m/s | velocidad mínima: el costo se topa en 1000 s/m | numérico |
| `aforo` | 6000 personas | gente al inicio, repartida parejo: 0.27 p/m² en 22,467 m² libres | escenario |
| salidas | 44 m de puerta | cada metro drena a lo más `Q_MAX` | escenario |
| fin | menos de 0.5 % dentro | se da por evacuado con menos de 30 personas | criterio nuestro |
| cuadros | cada 5 pasos = 1.5 s | lo que se guarda para los GIFs | solo visualización |

**Cota para comparar:** aunque todos estuvieran pegados a las puertas, por 44 m de salida pasan a lo más 44 × 1.2249 = 53.9 p/s, así que 6000 personas tardan al menos **111 s**. La simulación da 193 s; la diferencia es caminar hasta la puerta y la congestión.

*Weidmann, U. (1993). Transporttechnik der Fussgänger. Schriftenreihe des IVT Nr. 90, ETH Zürich.*

In [ ]:
aforo = 6000
M = np.where(~ocupado, aforo / (~ocupado).sum(), 0.0)    # gente repartida parejo
dt, cada = 0.3, 8
cuadros, tiempos = [], []
for n in range(20000):
    if n % cada == 0:
        T = campo(salidas, ocupado, weidmann(M))          # el terreno se rehace con la densidad
    if n % 5 == 0:
        cuadros.append(M.copy()); tiempos.append(n * dt)
    M, _ = paso(M, T, dt)
    if M.sum() <= 0.005 * aforo:                          # queda menos del 0.5 %: evacuado
        break
print("tiempo de evacuación: %.1f s" % ((n + 1) * dt))

### ¿Dónde estorba la multitud?

$V_0 T - D$ = metros de más que le cuesta a cada celda salir por culpa de la gente. Es 0 donde la multitud no estorba.

Fíjate que el retraso no está solo donde hay gente: lo paga cualquier celda cuya ruta cruce una multitud. Por eso el cuarto de arriba a la izquierda sale como el peor: para salir hay que cruzar el tapón de su puerta de 2 m.

In [ ]:
import matplotlib.colors as mcolors

k = 25                                               # el cuadro de los 37.5 s
T_k = campo(salidas, ocupado, weidmann(cuadros[k]))  # el T que usaba la simulación en ese momento
extra = V0 * T_k - D                                 # D: el de la celda del terreno (velocidad 1)
extra[~np.isfinite(extra)] = np.nan                  # muros y módulos fuera

fig, ax = plt.subplots(1, 2, figsize=(16, 4.8))
im = ax[0].imshow(cuadros[k], origin="lower", extent=(0, W, 0, H), cmap="YlOrRd",
                  norm=mcolors.PowerNorm(gamma=0.4, vmin=0, vmax=RHO_MAX))
ax[0].set_title("densidad a los %.0f s" % tiempos[k])
plt.colorbar(im, ax=ax[0], label="personas / m²")
im = ax[1].imshow(extra, origin="lower", extent=(0, W, 0, H), cmap="magma_r")
ax[1].set_title("V0·T − D: metros de más por la multitud")
plt.colorbar(im, ax=ax[1], label="m")
plt.tight_layout()
plt.show()
print("donde más estorba: %.0f m de más (%.0f s)" % (np.nanmax(extra), np.nanmax(extra) / V0))

In [ ]:
import os
import matplotlib.colors as mcolors
from matplotlib import animation
from matplotlib.patches import Patch, Rectangle
import IPython.display as ipd

os.makedirs("graficas", exist_ok=True)

cuerpo = np.zeros((H, W), dtype=bool)                # solo los módulos
for m in layout:
    pintar(cuerpo, m.x, m.y, m.ancho, m.fondo)

# escala de color: gris = ya no hay nadie; de amarillo a rojo oscuro = de poca gente al atasco
escala = [[0, "#d9d9d9"], [0.03, "#ffffcc"], [0.35, "#feb24c"], [0.7, "#f03b20"], [1, "#800026"]]
mapa = mcolors.LinearSegmentedColormap.from_list("densidad", escala)
norma = mcolors.PowerNorm(gamma=0.4, vmin=0, vmax=RHO_MAX)   # ** 0.4 realza las densidades bajas
marcas = [0, 0.1, 0.5, 1, 2, 4]                      # densidades reales que se escriben en la barra

# lo que no se mueve, en una capa encima: muros grises, puertas negras y salidas verdes
fijo = np.zeros((H, W, 4))                           # RGBA; alfa 0 = transparente
fijo[muros] = mcolors.to_rgba("#9e9e9e")
fijo[salidas] = mcolors.to_rgba("lime")
for m in layout:
    for x, y in m.celdas_puerta():
        fijo[y, x] = mcolors.to_rgba("black")

fig, ax = plt.subplots(figsize=(11, 6.5))
im = ax.imshow(cuadros[0], origin="lower", extent=(0, W, 0, H), cmap=mapa, norm=norma)
ax.imshow(fijo, origin="lower", extent=(0, W, 0, H))
for m in layout:                                     # módulos con contorno: si no, la comida se pierde en el naranja
    ax.add_patch(Rectangle((m.x, m.y), m.ancho, m.fondo, facecolor=m.tipo.color, edgecolor="black", lw=0.8))

barra = fig.colorbar(im, ax=ax, shrink=0.8, ticks=marcas, label="personas / m²")
barra.ax.set_yticklabels(["0 (nadie)"] + [str(d) for d in marcas[1:]])
tipos = list(dict.fromkeys(m.tipo for m in layout))  # cada tipo una sola vez
ax.legend(handles=[Patch(facecolor=t.color, edgecolor="black", label=t.nombre) for t in tipos]
                  + [Patch(color="#9e9e9e", label="Muros"), Patch(color="black", label="Puertas"),
                     Patch(color="lime", label="Salidas")],
          loc="upper center", bbox_to_anchor=(0.5, -0.09), ncol=7, fontsize=9, frameon=False)
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
titulo = ax.set_title("")
fig.tight_layout()

def cuadro(i):
    im.set_data(cuadros[i])                          # solo cambia la densidad; lo demás se queda
    titulo.set_text("t = %.0f s   |   dentro: %.0f personas" % (tiempos[i], cuadros[i].sum()))

anim = animation.FuncAnimation(fig, cuadro, frames=len(cuadros))
anim.save("graficas/planta.gif", writer=animation.PillowWriter(fps=10))   # 1 cuadro = 1.5 s: 15 veces más rápido
plt.close(fig)                                       # que no salga además como imagen fija
ipd.display(ipd.Image(filename="graficas/planta.gif"))

In [ ]:
import plotly.io as pio
from PIL import Image

tope = RHO_MAX ** 0.4
techo = tope * 0.5                                   # muros y módulos bajitos: en una vista fija no deben tapar a la gente

def altura(M):
    z = (M ** 0.4).astype(np.float32)                # mismo realce que el color: el inicio ya tiene altura
    z[muros | cuerpo] = np.nan                       # muros y módulos van como cajas
    return z

def escalera(A):
    """Cada celda pasa a 2 x 2 vértices con su mismo valor: arriba plana, lados verticales.
    Así se dibuja lo que de verdad calcula el modelo: un valor constante por celda."""
    return np.repeat(np.repeat(A, 2, axis=0), 2, axis=1)

bx = np.column_stack([np.arange(W), np.arange(W) + 1]).ravel()   # 0,1, 1,2, 2,3 ... los dos bordes de cada celda
by = np.column_stack([np.arange(H), np.arange(H) + 1]).ravel()

def cubos(mascara, alto, color, nombre):
    """Una caja de 1 x 1 m por cada celda marcada, todas en un solo trazo."""
    ys, xs = np.nonzero(mascara)
    esq_x = np.array([0, 1, 1, 0, 0, 1, 1, 0])       # las 8 esquinas de un cubo de 1 m
    esq_y = np.array([0, 0, 1, 1, 0, 0, 1, 1])
    esq_z = np.array([0, 0, 0, 0, 1, 1, 1, 1])
    caras = np.array([[0, 1, 2], [0, 2, 3], [4, 5, 6], [4, 6, 7],    # abajo, arriba
                      [0, 1, 5], [0, 5, 4], [1, 2, 6], [1, 6, 5],    # sur, este
                      [2, 3, 7], [2, 7, 6], [3, 0, 4], [3, 4, 7]])   # norte, oeste
    t = (caras[None] + 8 * np.arange(len(xs))[:, None, None]).reshape(-1, 3)   # cada cubo usa sus 8 esquinas
    alto = np.broadcast_to(alto, mascara.shape)[ys, xs]   # un número para todas, o una altura por celda
    return go.Mesh3d(x=(xs[:, None] + esq_x).ravel(), y=(ys[:, None] + esq_y).ravel(),
                     z=(alto[:, None] * esq_z).ravel(), i=t[:, 0], j=t[:, 1], k=t[:, 2],
                     color=color, flatshading=True, name=nombre, showlegend=True)

barra = dict(title="personas / m²", tickvals=[d ** 0.4 for d in marcas],
             ticktext=["0 (nadie)"] + [str(d) for d in marcas[1:]], thickness=14, len=0.6, x=0.92)

fig = go.Figure(go.Surface(x=bx, y=by, z=escalera(altura(cuadros[0])),   # el color sale de la misma altura
                           colorscale=escala, cmin=0, cmax=tope, colorbar=barra))
fig.add_trace(cubos(muros, techo, "#9e9e9e", "Muros"))
vistos = set()
for m in layout:
    c = caja(m, techo)
    c.update(showlegend=m.tipo not in vistos)        # una entrada por tipo en la leyenda
    vistos.add(m.tipo)
    fig.add_trace(c)
ys, xs = np.nonzero(salidas)
fig.add_scatter3d(x=xs + 0.5, y=ys + 0.5, z=np.zeros(len(xs)), mode="markers",
                  marker=dict(color="lime", size=3), name="Salidas")
fig.update_layout(margin=dict(l=0, r=0, t=40, b=0),
                  title=dict(x=0.5), legend=dict(x=0.01, y=0.95),
                  scene=dict(aspectmode="manual", aspectratio=dict(x=2.7, y=1.62, z=0.54),   # más grande = más zoom
                             xaxis_title="x (m)", yaxis_title="y (m)",
                             zaxis=dict(range=[0, tope * 1.02], visible=False)),   # la altura ya la explica la barra
                  scene_camera=dict(projection=dict(type="orthographic"),   # isométrica: sin perspectiva
                                    eye=dict(x=-0.95, y=-1.2, z=0.9)))       # desde la esquina suroeste

# una figura por cuadro: misma escena, solo cambia la altura y el título
sel = list(range(0, len(cuadros), 3))                # uno de cada 3: 4.5 s de simulación por cuadro
figs = []
for i in sel:
    f = go.Figure(fig)
    f.data[0].z = escalera(altura(cuadros[i]))
    f.update_layout(title_text="t = %.0f s   |   dentro: %.0f personas" % (tiempos[i], cuadros[i].sum()))
    figs.append(f)

def a_gif(figs, nombre, ms=300):
    """Saca una foto de cada figura con kaleido y las junta en graficas/<nombre>.gif."""
    os.makedirs("graficas/" + nombre, exist_ok=True)
    archivos = ["graficas/%s/%03d.png" % (nombre, k) for k in range(len(figs))]
    pio.write_images(figs, archivos, width=1100, height=580)
    fotos = [Image.open(a).convert("RGB") for a in archivos]
    # una sola paleta para todos los cuadros (si no, los colores parpadean)
    muestra = Image.new("RGB", (fotos[0].width, 3 * fotos[0].height))
    for k, f in enumerate([fotos[0], fotos[len(fotos) // 2], fotos[-1]]):
        muestra.paste(f, (0, k * f.height))
    paleta = muestra.quantize(256)
    gif = [f.quantize(palette=paleta, dither=Image.Dither.NONE) for f in fotos]
    gif[0].save("graficas/%s.gif" % nombre, save_all=True, append_images=gif[1:], duration=ms, loop=0)
    ipd.display(ipd.Image(filename="graficas/%s.gif" % nombre))

a_gif(figs, "isometrica")                            # 300 ms por cuadro: la misma velocidad que la planta

### El recinto se vacía como un líquido

Piso gris = **D**, el terreno de las canicas. Encima, la densidad de cada celda como altura extra:

$$z = D + K\rho, \qquad K = 4 \text{ m por p/m²}$$

Así el "volumen" de líquido es proporcional a la gente que queda: cuando se vacía, se vacía de verdad.

El comportamiento es un poco raro para un líquido, y es lo interesante:
- El agua escurre según la pendiente de su propia superficie y se nivela. Aquí no: la gente baja según **T**, que depende de la densidad de *todas* las celdas, no del piso D que se dibuja.
- Por eso se amontona frente a las puertas en lugar de nivelarse, y puede cruzar una "cresta" de D para irse a otra salida menos llena: elige el camino más rápido, no el de más pendiente.

In [ ]:
from scipy import ndimage

K = 4                                                # 1 persona/m² = 4 m de "líquido" encima del piso
bloqueado = muros | cuerpo
ok = np.isfinite(D)

# el piso debajo de muros y módulos: el D de la celda libre más cercana
_, (iy, ix) = ndimage.distance_transform_edt(~ok, return_indices=True)
piso = D[iy, ix]
# cada muro llega 3 m arriba del piso más alto que lo toca: se ve como bordo sobre el terreno
borde = np.maximum(ndimage.maximum_filter(np.where(ok, D, -np.inf), size=3), piso) + 3

def nivel(M):
    z = (piso + K * M).astype(np.float32)            # piso gris (D) + la capa de gente encima
    z[bloqueado] = np.nan                            # muros y módulos van como cajas
    return z

def color(M):
    return (M ** 0.4).astype(np.float32)             # el mismo realce de siempre: lo poco también se ve

fig = go.Figure(go.Surface(x=bx, y=by, z=escalera(nivel(cuadros[0])), surfacecolor=escalera(color(cuadros[0])),
                           colorscale=escala, cmin=0, cmax=tope, colorbar=barra))
fig.add_trace(cubos(muros, borde, "#9e9e9e", "Muros"))
vistos = set()
for m in layout:
    c = caja(m, borde[m.y:m.y + m.fondo, m.x:m.x + m.ancho].max())
    c.update(showlegend=m.tipo not in vistos)
    vistos.add(m.tipo)
    fig.add_trace(c)
ys, xs = np.nonzero(salidas)
fig.add_scatter3d(x=xs + 0.5, y=ys + 0.5, z=np.zeros(len(xs)), mode="markers",
                  marker=dict(color="lime", size=3), name="Salidas")
fig.update_layout(margin=dict(l=0, r=0, t=40, b=0),
                  title=dict(x=0.5), legend=dict(x=0.01, y=0.95),
                  scene=dict(aspectmode="manual", aspectratio=dict(x=2.4, y=1.44, z=0.7),
                             xaxis_title="x (m)", yaxis_title="y (m)", zaxis_visible=False),
                  scene_camera=dict(projection=dict(type="orthographic"),
                                    eye=dict(x=-0.95, y=-1.2, z=0.9)))

figs = []
for i in sel:
    f = go.Figure(fig)
    f.data[0].update(z=escalera(nivel(cuadros[i])), surfacecolor=escalera(color(cuadros[i])))
    f.update_layout(title_text="t = %.0f s   |   dentro: %.0f personas" % (tiempos[i], cuadros[i].sum()))
    figs.append(f)
a_gif(figs, "liquido")